# Colab: tuning and training (Stages 3-4)

I did not use this notebook for any result in the repository — the full grid
search ran in about 40 minutes on my laptop's CPU. I keep it here as a working
GPU setup in case the search grid is ever expanded enough to need one.

Stages 1-2 run locally first. Ingest needs the full raw dataset (tens of GB)
and is disk-bound rather than GPU-bound, so uploading it here would waste hours
and would not fit in a free 15 GB Drive. What gets uploaded instead is the
single processed file `internet_matrix.npy` (~341 MB) plus three tiny metadata
files, which is all the models need.

Before starting: **Runtime -> Change runtime type -> T4 GPU**.

In [ ]:
# 1. Confirm a GPU is actually attached. If this errors, fix the runtime type.!nvidia-smi

## 2. Mount Drive

The four processed files go in `MyDrive/milan/processed/` before this runs:

```
internet_matrix.npy      <- the only large one (~341 MB)
timestamps_ns.npy
square_ids.npy
meta.json
```

They are produced locally by `make data`, in `data/processed/`.
`observed_mask.npy` is not needed for training.

In [ ]:
from google.colab import drivedrive.mount('/content/drive')DRIVE = '/content/drive/MyDrive/milan'!mkdir -p {DRIVE}/processed {DRIVE}/results!ls -lh {DRIVE}/processed

## 3. Clone the repository and install dependencies

In [ ]:
REPO = 'https://github.com/Eddydev-ALU/milan-traffic-forecasting.git'%cd /content![ -d milan-traffic-forecasting ] || git clone $REPO%cd /content/milan-traffic-forecasting!git pull --quiet || true# torch is preinstalled on Colab; statsmodels usually needs an upgrade.!pip install -q --upgrade statsmodels pyyaml psutilimport torchprint('torch', torch.__version__, '| cuda available:', torch.cuda.is_available())

In [ ]:
# Sanity check before spending GPU time.!python -m pytest tests/ -q

## 4. Colab config

Data is read from Drive and results are written back to Drive, so a disconnect
loses nothing. Absolute paths in the config override the project root, which is
what makes this work.

I edit `sequence_length` and the `tuning` grids here rather than in
`configs/config.yaml`, so the local config stays untouched.

In [ ]:
config_text = f'''seed: 42paths:  raw_dir: data/raw  processed_dir: {DRIVE}/processed  results_dir: {DRIVE}/results  figures_dir: {DRIVE}/results/figures  tables_dir: {DRIVE}/results/tables  predictions_dir: {DRIVE}/results/predictionsdataset:  activity: internet  n_squares: 10000  intervals_per_day: 144  timezone: Europe/Rome  chunksize: 2000000  missing_policy: interpolatesplits:  train_start: "2013-11-01 00:00"  train_end:   "2013-12-08 23:50"  val_start:   "2013-12-09 00:00"  val_end:     "2013-12-15 23:50"  test_start:  "2013-12-16 00:00"  test_end:    "2013-12-22 23:50"eda:  extra_squares: [4159, 4556]  first_two_weeks_start: "2013-11-01 00:00"  first_two_weeks_end:   "2013-11-14 23:50"  acf_max_lag: 1100forecasting:  horizon: 1  sequence_length: 144  use_time_features: true  scaler:    log1p: true    method: standardmodels:  baselines:    seasonal_period: 144  sarimax:    order: [2, 0, 1]    fourier:      - {{period: 144,  n_terms: 5}}      - {{period: 1008, n_terms: 3}}    log1p: true    trend: c    maxiter: 200  lstm:    hidden_size: 64    num_layers: 2    dropout: 0.2    lr: 0.001    batch_size: 128    max_epochs: 60    patience: 8    grad_clip: 1.0    loss: huber  tcn:    channels: [32, 32, 32, 32]    kernel_size: 3    dropout: 0.15    lr: 0.002    batch_size: 128    max_epochs: 60    patience: 8    grad_clip: 1.0    loss: hubertuning:  lstm:    sequence_length: [24, 144, 288]    hidden_size: [32, 64, 128]    num_layers: [1, 2]    lr: [0.003, 0.001]  tcn:    sequence_length: [144, 288]    kernel_size: [3, 5]    dropout: [0.1, 0.2]    lr: [0.003, 0.001]  sarimax:    order: [[1, 0, 0], [2, 0, 1], [3, 0, 2], [1, 1, 1]]timing:  n_repeats: 3  warmup: 1'''open('configs/config_colab.yaml', 'w').write(config_text)print('written')

## 5. Which square am I tuning on?

Stage 2 caches the top three squares to `results/tables/top3.json`. That file
goes in `{DRIVE}/results/tables/`, or the IDs can be set by hand below.

In [ ]:
import json, ostop3_path = f'{DRIVE}/results/tables/top3.json'if os.path.exists(top3_path):    print('top 3 areas:', json.load(open(top3_path))['top3'])else:    print('top3.json not found -- upload it, or pass --squares explicitly below.')

## 6. Grid search

Safe to re-run after a disconnect. Every trial is flushed to
`experiment_log.csv` on Drive as it completes, and completed trials are skipped
on the next run. If the session drops at trial 20 of 36, running this cell again
picks up where it left off.

The full LSTM grid is 36 configurations. `--max-trials 8` first gives a read on
the per-trial cost before committing to the whole sweep.

In [ ]:
!python scripts/03_run_experiments.py --config configs/config_colab.yaml --tune --max-trials 8

In [ ]:
# Once you know the pace, run the rest. Re-run freely; it resumes.!python scripts/03_run_experiments.py --config configs/config_colab.yaml --tune

In [ ]:
import pandas as pd
log = pd.read_csv(f'{DRIVE}/results/tables/experiment_log.csv')
display(log.sort_values('val_MAE').head(15))

## 7. Final runs

The winning hyperparameters go into the `models:` block of
`configs/config_colab.yaml` above; re-run that cell, then run this.

In [ ]:
!python scripts/03_run_experiments.py --config configs/config_colab.yaml

In [ ]:
!python scripts/04_failure_analysis.py --config configs/config_colab.yaml

## 8. Collect the results

Everything is already on Drive, but a zip is easier to pull down in one go.

In [ ]:
!cd {DRIVE} && zip -rq results_colab.zip results && ls -lh {DRIVE}/results_colab.zipfrom google.colab import filesfiles.download(f'{DRIVE}/results_colab.zip')

## Notes

- **Keep the tab open.** Free Colab disconnects after ~90 minutes idle and caps
  sessions at ~12 hours. Resume support covers a dropped session, but it cannot
  resume a single trial that was mid-flight.
- **SARIMAX gets no GPU benefit.** `statsmodels` is CPU-only, so that stage is
  better left on a local machine.
- **If the GPU limit is hit**, the same commands run on a CPU runtime, just
  slower. The TCN is far less affected than the LSTM, because convolutions
  parallelise over the time axis while recurrence does not.
- **Timings are hardware-specific.** Times measured on a T4 are not comparable
  with times measured on a laptop CPU, so a run that mixes the two cannot be
  reported as a single comparison. Every timing in this repository came from one
  CPU-only machine.